# fraud-v3-candidate — shadow run investigation

The model platform blocked the promotion (TESS-2310).

| | AUC | PR-AUC |
|---|---|---|
| candidate, offline (notebook 01) | **0.906** | 0.358 |
| candidate, shadow (2026-07-05 → 2026-08-27) | **0.699** | 0.071 |
| champion fraud-v2, same window | 0.778 | 0.164 |

Offline it beat the champion by a wide margin. In shadow it is *worse* than the champion. Why?

In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import average_precision_score, roc_auc_score

ROOT = Path.cwd().resolve()
while not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
print(ROOT)

In [ ]:
report = json.loads((ROOT / "ml/registry/shadow/fraud-v3-candidate.json").read_text())
pd.DataFrame({
    "offline": report["offline"],
    "shadow": {k: v for k, v in report["shadow"].items() if k != "by_month"},
    "champion": report["champion_same_window"],
}).T

In [ ]:
df = pd.read_parquet(ROOT / "ml/data/transactions.parquet").sort_values("timestamp").reset_index(drop=True)
cut = df.timestamp.quantile(0.70)          # same temporal split as notebook 01
train, test = df[df.timestamp <= cut].copy(), df[df.timestamp > cut].copy()
print(f"train={len(train):,}  test={len(test):,}  test window {test.timestamp.min().date()} → {test.timestamp.max().date()}")
print("the shadow window is the test window")

## Questions

1. Can the offline number be reproduced from notebook 01's feature set?
2. The report says `card_chargeback_rate` is non-zero for 27.4% of training rows but 14.2% of scored rows. Does the feature look the same at training time as at scoring time?
3. What does the candidate score without it?